# Collecting data from web sources
Source: Wikipedia, *List of countries by exports* (text licensed CC BY-SA 4.0; the underlying figures are from the World Bank).
If the internet is unavailable, use the cached copy in `data/cache/wikipedia_exports.html`.

## 1. Are we allowed? Check robots.txt

In [ ]:
import requests
from urllib import robotparser

URL = ("https://en.wikipedia.org/wiki/"
       "List_of_countries_by_exports")
HEADERS = {"User-Agent":
           "TradeCourse/1.0 (me@example.org)"}
robots = requests.get(
    "https://en.wikipedia.org/robots.txt",
    headers=HEADERS, timeout=30)
rp = robotparser.RobotFileParser()
rp.parse(robots.text.splitlines())
print(rp.can_fetch("*", URL))

## 2. Download the page

In [ ]:
resp = requests.get(URL, headers=HEADERS,
                    timeout=30)
print(resp.status_code)
print(resp.headers["Content-Type"])
html = resp.text
print(len(html), "characters")

## 3. Parse the HTML

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")
print(soup.title.get_text())

table = soup.find("table", class_="wikitable")
caption = table.find("caption")
print(caption.get_text(strip=True))

## 4. Extract the rows

In [ ]:
rows = []
for tr in table.find_all("tr")[1:]:
    tds = tr.find_all(["th", "td"])
    rows.append([c.get_text(strip=True)
                 for c in tds])
print(len(rows))
print(rows[:3])

## 5. Into pandas, with types fixed

In [ ]:
import pandas as pd

df = pd.DataFrame(rows, columns=[
    "country", "exports_usd_m", "year",
    "top_export"])
df["exports_usd_m"] = (
    df["exports_usd_m"]
    .str.replace(",", "")
    .astype(float))
df["year"] = df["year"].astype(int)
df.head()

## 6. The shortcut: pandas.read_html

In [ ]:
from io import StringIO

tables = pd.read_html(StringIO(html),
                      match="Exports")
print(len(tables), "matching tables")
tables[0].head()

## 7. Data quality: not every row is the same year

In [ ]:
print(df["year"].value_counts().head())
old = df[df["year"] < df["year"].max()]
print(len(old), "countries with older data")
old.head()

## Offline fallback

In [ ]:
from pathlib import Path

CACHE = Path("../../data/cache")
cached = CACHE / "wikipedia_exports.html"
html = cached.read_text(encoding="utf-8")
print(len(html), "characters from cache")